# Capítulo 2 · Ejemplos y razonamiento paso a paso

Un clasificador de correos con y sin ejemplos en la instrucción, medido con la exactitud y la matriz de confusión, y el cálculo de reembolsos respondiendo directamente, razonando paso a paso, con un ejemplo resuelto y votando entre varias respuestas.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. Con GPU, el cuaderno usa Qwen2.5-1.5B-Instruct; sin ella, Qwen3-0.6B. La clasificación tarda alrededor de un minuto; los reembolsos, bastante más, porque son setenta respuestas razonadas.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

📘 El libro, en Amazon: [tapa blanda](https://www.amazon.es/dp/B0HMXCBCTN) · [Kindle](https://www.amazon.es/dp/B0HMXG3NXM)

## 1. El modelo y los datos

La primera vez se descarga el modelo y el archivo con los correos etiquetados del repositorio del libro.

In [ ]:
"""Capítulo 2: ejemplos y razonamiento paso a paso. Modelo abierto, en GPU o CPU."""
import json
import re
import time
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-0.6B"             # sin GPU: más pequeño y más rápido
MODELO = None                              # None: elegir según el entorno
SEMILLA = 0
DATOS = Path("datos") / "correos_clasificados.json"
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"



def cargar(nombre=None):
    """Carga el modelo y su tokenizador en la GPU si la hay; si no, en la CPU."""
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    if nombre is None:
        nombre = MODELO_GPU if dispositivo == "cuda" else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    modelo.to(dispositivo).eval()
    print(f"Modelo {nombre} en {dispositivo}")
    return tok, modelo


def cargar_correos():
    """Los correos etiquetados; en Colab, se descargan del repositorio."""
    if not DATOS.exists():
        DATOS.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + DATOS.as_posix(), DATOS)
    return json.loads(DATOS.read_text(encoding="utf-8"))

In [ ]:
tok, modelo = cargar(MODELO)
datos = cargar_correos()
print(len(datos["ejemplos"]), "ejemplos y", len(datos["prueba"]), "correos de prueba")
for categoria, definicion in datos["categorias"].items():
    print(f"{categoria}: {definicion}")

## 2. Preguntar con ejemplos

`mensajes` añade los ejemplos como turnos anteriores de la conversación.

In [ ]:
def mensajes(sistema, usuario, ejemplos=()):
    """Sistema, ejemplos resueltos (pares pregunta-respuesta) y la pregunta."""
    lista = [{"role": "system", "content": sistema}] if sistema else []
    for pregunta, respuesta in ejemplos:
        lista.append({"role": "user", "content": pregunta})
        lista.append({"role": "assistant", "content": respuesta})
    return lista + [{"role": "user", "content": usuario}]


def preguntar(sistema, usuario, ejemplos=(), temperatura=0, max_tokens=200):
    """Envía una conversación al modelo y devuelve su respuesta como texto."""
    texto = tok.apply_chat_template(mensajes(sistema, usuario, ejemplos),
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    if temperatura > 0:
        sorteo = dict(do_sample=True, temperature=temperatura, top_p=1.0, top_k=0)
    else:
        sorteo = dict(do_sample=False, temperature=None, top_p=None, top_k=None)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 pad_token_id=tok.eos_token_id, **sorteo)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    respuesta = tok.decode(nuevos, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S).strip()

In [ ]:
for m in mensajes("Clasifica el correo.", "¿Abrís los sábados?", datos["ejemplos"][:2]):
    print(m["role"], "→", m["content"])

## 3. Clasificar correos

In [ ]:
def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar sin preocuparse de ellas."""
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower())
                   if unicodedata.category(c) != "Mn")


def leer_categoria(respuesta, categorias):
    """La primera categoría que aparece en la respuesta, o None si no hay ninguna."""
    texto = sin_tildes(respuesta)
    posiciones = {c: texto.find(c) for c in categorias if c in texto}
    return min(posiciones, key=posiciones.get) if posiciones else None


def instruccion(datos, definiciones):
    """El mensaje de sistema: la tarea y las categorías, con o sin definición."""
    lineas = [f"- {c}: {d}" if definiciones else f"- {c}"
              for c, d in datos["categorias"].items()]
    return ("Clasifica el correo de un cliente de Ciclos Norte, una tienda de "
            "bicicletas, en una de estas categorías:\n" + "\n".join(lineas)
            + "\nResponde solo con el nombre de la categoría, sin tildes.")


def clasificar(datos, definiciones=True, ejemplos=()):
    """Clasifica los correos de prueba y devuelve los pares (correcta, predicha)."""
    sistema = instruccion(datos, definiciones)
    pares = []
    for correo, correcta in datos["prueba"]:
        r = preguntar(sistema, correo, ejemplos, max_tokens=10)
        pares.append((correcta, leer_categoria(r, datos["categorias"])))
    return pares


def exactitud(pares):
    """Proporción de correos bien clasificados."""
    return sum(c == p for c, p in pares) / len(pares)


def matriz_confusion(pares, categorias):
    """Filas: categoría correcta; columnas: predicha (la última, «ninguna»)."""
    columnas = list(categorias) + [None]
    matriz = [[0] * len(columnas) for _ in categorias]
    for correcta, predicha in pares:
        matriz[list(categorias).index(correcta)][columnas.index(predicha)] += 1
    return matriz

In [ ]:
print(instruccion(datos, definiciones=True))

In [ ]:
def probar_clasificador(datos):
    """Exactitud del clasificador con cuatro versiones de la instrucción."""
    ejemplos = datos["ejemplos"]
    una_por_categoria = ejemplos[::2]          # el primer ejemplo de cada categoría
    variantes = {
        "Solo las categorías": dict(definiciones=False),
        "Con definiciones": dict(definiciones=True),
        "Con 5 ejemplos": dict(definiciones=True, ejemplos=una_por_categoria),
        "Con 10 ejemplos": dict(definiciones=True, ejemplos=ejemplos),
    }
    resultados = {}
    for nombre, opciones in variantes.items():
        inicio = time.time()
        pares = clasificar(datos, **opciones)
        resultados[nombre] = pares
        print(f"{nombre:22s} exactitud {exactitud(pares):5.1%}  "
              f"({time.time() - inicio:.0f} s)")
    return resultados


def mostrar_matriz(pares, categorias):
    """Imprime la matriz de confusión."""
    matriz = matriz_confusion(pares, categorias)
    print(" " * 11 + "".join(f"{c[:8]:>9s}" for c in categorias) + "  ninguna")
    for categoria, fila in zip(categorias, matriz):
        print(f"{categoria:>11s}" + "".join(f"{n:>9d}" for n in fila))

In [ ]:
clasificacion = probar_clasificador(datos)

In [ ]:
mostrar_matriz(clasificacion["Solo las categorías"], datos["categorias"])

# Los correos mal clasificados con cada instrucción
for variante, pares in clasificacion.items():
    print("\n==", variante)
    for (correcta, predicha), (correo, _) in zip(pares, datos["prueba"]):
        if correcta != predicha:
            print(f"{correcta} → {predicha}: {correo}")

## 4. Calcular reembolsos

Diez problemas con su respuesta correcta y cuatro formas de pedir el cálculo.

In [ ]:
PROBLEMAS = [
    ("Compré unas ruedas de 120 euros y una bomba de 30 euros. Devuelvo la "
     "bomba y me cobran 5 euros por la recogida. ¿Cuánto me devolvéis?", 25),
    ("Compré un casco de 80 euros con un 25 % de descuento. Lo devuelvo. "
     "¿Cuánto me devolvéis?", 60),
    ("Compré dos maillots de 35 euros cada uno y unas zapatillas de 90 euros, "
     "con un 10 % de descuento en todo el pedido. Devuelvo los dos maillots. "
     "¿Cuánto me devolvéis?", 63),
    ("Compré tres cámaras de 8 euros cada una y un sillín de 46 euros. "
     "Devuelvo el sillín y una cámara. ¿Cuánto me devolvéis?", 54),
    ("Compré una bicicleta de 640 euros y un candado de 40 euros, con un 10 % "
     "de descuento en todo. Devuelvo el candado. ¿Cuánto me devolvéis?", 36),
    ("Compré 4 bidones a 7 euros cada uno. Devuelvo la mitad. ¿Cuánto me "
     "devolvéis?", 14),
    ("Compré una bicicleta eléctrica de 1500 euros y pagué 15 euros de envío. "
     "Llegó rota, así que la devuelvo y me reembolsáis también el envío. "
     "¿Cuánto me devolvéis?", 1515),
    ("Compré unos guantes de 24 euros y unas gafas de 56 euros, con un 25 % "
     "de descuento en todo. Devuelvo las gafas. ¿Cuánto me devolvéis?", 42),
    ("Compré un portabultos de 50 euros y unas alforjas de 150 euros, y usé un "
     "vale de 40 euros que se reparte entre los productos en proporción a su "
     "precio. Devuelvo las alforjas. ¿Cuánto me devolvéis?", 120),
    ("Compré 4 pares de calcetines a 10 euros y un culotte de 50 euros, con un "
     "10 % de descuento en todo. Devuelvo 2 pares de calcetines y el culotte. "
     "¿Cuánto me devolvéis?", 63),
]

In [ ]:
SISTEMA_REEMBOLSO = ("Eres el asistente de Ciclos Norte. Calcula cuánto "
                     "dinero hay que devolver al cliente.")


DIRECTA = "Responde solo con la cantidad en euros, sin explicaciones."


PASO_A_PASO = ("Razona paso a paso y termina con una última línea con el "
               "formato «Respuesta: <cantidad> euros».")


EJEMPLO_RESUELTO = [(
    "Compré un sillín de 40 euros y una luz de 20 euros, con un 20 % de "
    "descuento en todo. Devuelvo la luz. ¿Cuánto me devolvéis?",
    "La luz costaba 20 euros. Con el 20 % de descuento, por ella pagué "
    "20 × 0,80 = 16 euros, y eso es lo que se devuelve.\nRespuesta: 16 euros")]

In [ ]:
def leer_cantidad(respuesta):
    """La cantidad que sigue a «Respuesta:» o, si no la hay, el último número."""
    parte = respuesta.rsplit("Respuesta:", 1)[-1]
    numeros = re.findall(r"\d+(?:[.,]\d+)*", parte)
    if not numeros:
        return None
    numero = numeros[0] if "Respuesta:" in respuesta else numeros[-1]
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", numero):    # 1.515 son mil quinientos
        numero = numero.replace(".", "")
    return float(numero.replace(",", "."))


def reembolsar(problema, forma, ejemplos=(), temperatura=0):
    """Pide el cálculo con una forma de responder; devuelve el texto y la cantidad."""
    respuesta = preguntar(SISTEMA_REEMBOLSO + " " + forma, problema, ejemplos,
                          temperatura=temperatura, max_tokens=400)
    return respuesta, leer_cantidad(respuesta)


def votar(problema, veces=5):
    """Autoconsistencia: varias respuestas sorteadas; gana la más repetida."""
    cantidades = [reembolsar(problema, PASO_A_PASO, EJEMPLO_RESUELTO,
                             temperatura=0.7)[1] for _ in range(veces)]
    validas = [c for c in cantidades if c is not None]
    ganadora = Counter(validas).most_common(1)[0][0] if validas else None
    return ganadora, cantidades


def probar_reembolsos(veces=5):
    """Los diez problemas con las cuatro formas de pedir la respuesta."""
    torch.manual_seed(SEMILLA)
    filas = []
    for problema, correcta in PROBLEMAS:
        fila = {"problema": problema, "correcta": correcta}
        for clave, forma, ejemplos in (("directa", DIRECTA, ()),
                                       ("pasos", PASO_A_PASO, ()),
                                       ("ejemplo", PASO_A_PASO, EJEMPLO_RESUELTO)):
            fila["texto_" + clave], fila[clave] = reembolsar(problema, forma, ejemplos)
        fila["votacion"], fila["votos"] = votar(problema, veces)
        filas.append(fila)
        print(f"{correcta:>6} | directa {fila['directa']} | pasos {fila['pasos']}"
              f" | con ejemplo {fila['ejemplo']} | votación {fila['votacion']}",
              fila["votos"])
    for clave in ("directa", "pasos", "ejemplo", "votacion"):
        aciertos = sum(f[clave] == f["correcta"] for f in filas)
        print(f"Aciertos {clave}: {aciertos} de {len(filas)}")
    return filas

Antes de lanzar los diez problemas, mira cómo razona el modelo en uno de ellos, sin ejemplo y con él:

In [ ]:
problema = PROBLEMAS[1][0]
print(reembolsar(problema, PASO_A_PASO)[0])
print("\n--- Con el ejemplo resuelto:\n")
print(reembolsar(problema, PASO_A_PASO, EJEMPLO_RESUELTO)[0])

Ahora, los diez problemas de las cuatro formas. Es la parte lenta del cuaderno.

In [ ]:
reembolsos = probar_reembolsos()

## Ejercicios

1. Explica con tus palabras por qué se llama «aprendizaje» al uso de ejemplos en la instrucción, aunque el modelo no cambie. ¿En qué se parece y en qué se diferencia del entrenamiento?

2. En el archivo, los diez ejemplos están agrupados por categoría: primero los dos pedidos, luego las dos devoluciones, y así sucesivamente. Barájalos con `random.shuffle` y vuelve a medir la exactitud. ¿Cambia? ¿Y si los pones en orden inverso?

3. Modifica `clasificar` para que, si la respuesta no es ninguna de las categorías, vuelva a preguntar recordando la lista de categorías permitidas. ¿Desaparecen las respuestas inventadas con cinco ejemplos?

4. Escribe diez correos de prueba más difíciles: largos, con faltas de ortografía, que mezclen dos temas o escritos con ironía. ¿Cuánto baja la exactitud de la mejor instrucción? ¿Qué categorías se confunden más?

5. Uno de los correos de prueba, «¿Se puede pagar a plazos una bicicleta eléctrica?», confundió al modelo con las definiciones. ¿Te parece que la definición de *consulta* es clara? Mejórala y comprueba si cambia el resultado.

6. Con `mensajes` y el tokenizador, cuenta cuántos tokens añaden los diez ejemplos a cada petición. Si clasificaras 10 000 correos al mes con una API que cobra 0,50 euros por millón de tokens de entrada, ¿cuánto costarían los ejemplos?

7. Escribe un segundo ejemplo resuelto para los reembolsos, con un vale de descuento repartido en proporción al precio, y añádelo a `EJEMPLO_RESUELTO`. ¿Mejora el problema de las alforjas? ¿Y los demás?

8. Repite la votación con 11 intentos en lugar de 5. ¿Aumentan los aciertos? ¿Cuánto tarda?

9. Escribe una función `reembolso(precios, descuento, devueltos)` que haga el cálculo en Python. Después pide al modelo que extraiga de cada problema los precios, el descuento y los productos devueltos, en lugar de calcular. ¿Cuántos problemas resuelve así? Compara con la cadena de pensamiento.

10. Si tienes GPU, carga Qwen3-0.6B, cambia `enable_thinking=False` por `True` en `preguntar` y sube `max_tokens` a 1500. Resuelve los diez reembolsos con `DIRECTA`. ¿Cuántos acierta ahora? ¿Cuánto tarda cada respuesta? Lee alguno de sus razonamientos.

## Referencias

Modelos [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) y [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), de Qwen (Alibaba), con licencia Apache 2.0. Aprendizaje con pocos ejemplos: [Brown y colaboradores (2020)](https://arxiv.org/abs/2005.14165). Cadena de pensamiento: [Wei y colaboradores (2022)](https://arxiv.org/abs/2201.11903) y [Kojima y colaboradores (2022)](https://arxiv.org/abs/2205.11916). Autoconsistencia: [Wang y colaboradores (2022)](https://arxiv.org/abs/2203.11171). Lección 11.02 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte, sus correos y los problemas son ficticios. Código y experimentos propios.